# CloudSRE v2 — Kaggle Training (Headless)
Trains the cascade, multi-cascade, and adversarial tiers using GRPO.

In [ ]:
# Critical fix for unsloth/torchvision conflicts
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"

# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q httpx matplotlib wandb

In [ ]:
# Clone the repository
!git clone https://github.com/Harikishanth/CloudSRE.git
%cd CloudSRE

In [ ]:
# Kaggle Secrets Integration
try:
    from kaggle_secrets import UserSecretsClient
    import wandb
    
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    
    !huggingface-cli login --token $hf_token
    wandb.login(key=wandb_key)
    print("✅ Logged into HuggingFace and WandB")
except ImportError:
    print("Not running on Kaggle, skipping Kaggle secrets.")
except Exception as e:
    print("⚠️ Secrets not found. Please add HF_TOKEN and WANDB_API_KEY to Kaggle Secrets.")


## Phase 2: GRPO Training (Reinforcement Learning)

Resuming from Leg 1, now training on cascades and adversarial tiers.

In [ ]:
ENV_URL = "https://dardrax-cloudsre-environment.hf.space"

!python train_grpo.py \
    --env-url {ENV_URL} \
    --model-id DarDrax/cloudsre-1.5B-leg1 \
    --curriculum cascade,multi_cascade,adversarial \
    --episodes-per-tier 25 \
    --group-size 4 \
    --wandb-project CloudSRE-Hackathon-Run